In [8]:
import time
import requests
import pandas as pd

ENDPOINT = "https://query.wikidata.org/sparql"
HEADERS = {
    "User-Agent": "Trollownik/1.0 (mszostek@gmail.com)",
    "Accept": "application/sparql-results+json"
}

def sparql_query(query):
    response = requests.get(
        ENDPOINT,
        params={"query": query, "format": "json"},
        headers=HEADERS,
        timeout=120
    )
    response.raise_for_status()
    data = response.json()
    rows = []
    for result in data["results"]["bindings"]:
        rows.append({k: v["value"] for k, v in result.items()})
    df = pd.DataFrame(rows)
    
    for col in df.columns:
        if pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].str.replace(
                "http://www.wikidata.org/entity/", "wd:", regex=False
            )
        if "date" in col.lower():
            df[col] = df[col].str.extract(r"^(-?\d{1,4}-\d{2}-\d{2})", expand=False)
    return df

def gas_query(qid, link, traversal_direction):
    query = f"""
    SELECT DISTINCT ?item ?label ?description (GROUP_CONCAT(?jlabel; separator=", ") AS ?jobs) ?bdate ?bplacelabel ?ddate ?dplacelabel ?burialplace ?depth {{
      SERVICE gas:service {{
        gas:program gas:gasClass "com.bigdata.rdf.graph.analytics.BFS" ;
                    gas:in {qid} ;
                    gas:traversalDirection "{traversal_direction}" ;
                    gas:out ?item ;
                    gas:out1 ?depth ;
                    gas:maxIterations 5 ;
                    gas:linkType {link} .
      }}

      OPTIONAL {{ ?item wdt:P106 ?job. }}
      OPTIONAL {{ ?item wdt:P569 ?bdate.}} 
      OPTIONAL {{ ?item wdt:P570 ?ddate.}} 
      OPTIONAL {{ ?item wdt:P19 ?bplace.}} 
      OPTIONAL {{ ?item wdt:P20 ?dplace.}} 
      OPTIONAL {{ ?item wdt:P20 ?dplace.}} 
      OPTIONAL {{ ?item wdt:P119 ?burplace.}}
      
      SERVICE wikibase:label {{
        bd:serviceParam wikibase:language "pl,en" .
        ?item rdfs:label ?label. ?item schema:description ?description.
        ?job rdfs:label ?jlabel.
        ?king rdfs:label ?lking.
        ?jedn rdfs:label ?jlabel.
        ?bplace rdfs:label ?bplacelabel.
        ?dplace rdfs:label ?dplacelabel.
        ?burplace rdfs:label ?burialplace.
        ?king rdfs:label ?klabel.
      }}
    }} GROUP BY ?item ?label ?description ?bdate ?bplacelabel ?ddate ?dplacelabel ?burialplace ?depth
    """
    return sparql_query(query)

query_kings = """
SELECT DISTINCT ?king ?kingLabel {
  ?king p:P39 ?stmt.
  ?stmt ps:P39 wd:Q3273712.
#  FILTER NOT EXISTS { ?stmt wikibase:rank wikibase:DeprecatedRank. }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "pl,en". }
}
"""
kings = sparql_query(query_kings)
print(f"Found {len(kings)} kings")

directions = [
    ("descendant", "wdt:P40", "Forward"),
    # ("ancestor",   "(wdt:P22|wdt:P25)", "Forward"),  # father, grandfather...
    ("ancestor",   "wdt:P22", "Forward"),
    ("ancestor",   "wdt:P25", "Forward"), 
]

results = []

# i = 1

for _, row in kings.iterrows():
    qid = row["king"].replace("http://www.wikidata.org/entity/", "wd:")
    print(f"\nProcessing: {row['kingLabel']} ({row["king"]})")

    for relation, link, traversal in directions:
        print(f"  {relation} via {link}…")
        try:
            df = gas_query(qid, link, traversal)
            if not df.empty:
                df["king"] = row["kingLabel"]
                df["kingQID"] = row["king"]
                df["relation"] = relation
                df["linkType"] = link
                results.append(df)
        except Exception as e:
            print(f"  Error: {e}")

        time.sleep(.5)
    # i += 1
    # if i >= 3:
    #     break

df_all = pd.concat(results, ignore_index=True)

df_all = (df_all
    .groupby(["king", "kingQID", "item", "label", "description", "jobs", "bdate", "bplacelabel", "ddate", "dplacelabel", "burialplace", "depth", ])
    ["relation"]
    .agg(lambda x: "/".join(sorted(set(x))))
    .reset_index()
)

df_all.to_csv("kings_families.csv", index=False, sep="\t")
print(f"\nSaved {len(df_all)} rows to kings_families.csv")


Found 50 kings

Processing: Siemomysł (wd:Q161831)
  descendant via wdt:P40…
  ancestor via wdt:P22…
  ancestor via wdt:P25…

Processing: Zbigniew (wd:Q167992)
  descendant via wdt:P40…
  ancestor via wdt:P22…
  ancestor via wdt:P25…

Processing: Anna Jagiellonka (wd:Q233989)
  descendant via wdt:P40…
  ancestor via wdt:P22…
  ancestor via wdt:P25…

Processing: Wacław II (wd:Q155581)
  descendant via wdt:P40…
  ancestor via wdt:P22…
  ancestor via wdt:P25…

Processing: Aleksander III Romanow (wd:Q120180)
  descendant via wdt:P40…
  ancestor via wdt:P22…
  ancestor via wdt:P25…

Processing: Jan Luksemburski (wd:Q155167)
  descendant via wdt:P40…
  ancestor via wdt:P22…
  ancestor via wdt:P25…

Processing: Rudolf III Habsburg (wd:Q168254)
  descendant via wdt:P40…
  ancestor via wdt:P22…
  ancestor via wdt:P25…

Processing: Helena znojemska (wd:Q2128810)
  descendant via wdt:P40…
  ancestor via wdt:P22…
  ancestor via wdt:P25…

Processing: Dobroniega Maria (wd:Q263296)
  descendant via w